# Experiment: Norman Nonadditive Downstream Experiment

这个 notebook 对应 `scripts/trishift/analysis/norman_nonadd_experiment.py`。

目的：
- 检查组合扰动 `A+B` 是否被模型学成简单加和，还是恢复了非加性 interaction。
- 直接比较 truth non-additive residual 和 pred non-additive residual。

注意：这个 notebook 只支持 `dataset = "norman"`。


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "scripts").exists() and (repo_root.parent / "scripts").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import scripts.trishift.analysis.norman_nonadd_experiment as norman_nonadd_experiment
importlib.reload(norman_nonadd_experiment)

run_norman_nonadd_experiment = norman_nonadd_experiment.run_norman_nonadd_experiment

repo_root


## Parameters

默认先比较几种主模型；如果你只想对比 `TriShift nearest` 和 `Scouter`，直接在 `models` 里删掉别的即可。

`space` 可选：
- `full_gene`: 只用 `Pred_full / Truth_full / Ctrl_full`
- `deg`: 用 `Pred / Truth / Ctrl` 和共享 DE 基因交集
- `auto`: 优先 full-gene，没有时回退到 DEG


In [ ]:
dataset = "norman"
models = ["trishift_nearest", "scouter", "gears", "genepert"]
split_ids = [1]
out_root = None
space = "full_gene"  # full_gene | deg | auto

result = run_norman_nonadd_experiment(
    dataset=dataset,
    models=models,
    split_ids=split_ids,
    out_root=out_root,
    space=space,
)

print(f"out_dir: {result['out_dir']}")


## Summary Tables

先看总体 summary，再看 subgroup，最后检查哪些组合因为缺单基因条件被跳过。


In [ ]:
display(result["summary_df"])
display(result["subgroup_df"])
display(result["skipped_df"].head(30))
display(result["per_condition_df"].head(30))


## Plots

`nonadd_pearson` 和 `nonadd_r2` 越高越好；`interaction_strength_mae` 越低越好。


In [ ]:
out_dir = Path(result["out_dir"])
display(Image(filename=str(out_dir / "norman_nonadd_scatter.png")))
display(Image(filename=str(out_dir / "norman_nonadd_by_subgroup.png")))

for path in [
    out_dir / "norman_nonadd_per_condition.csv",
    out_dir / "norman_nonadd_summary.csv",
    out_dir / "norman_nonadd_by_subgroup.csv",
    out_dir / "norman_nonadd_skipped.csv",
    out_dir / "run_meta.json",
]:
    print(path)
